# MA: Batch 1 — Identity groups (notebooks first)

**Purpose:** Implement and verify the five Batch 1 actions in the notebook before any router code. MA order: math → lemmas → invariants → **notebooks** → code.

**Batch 1 actions:** `list_groups`, `get_group`, `create_group`, `list_group_members`, `add_group_member`.

**Sources:** `docs/contracts/caio-m365/ACTION_SPECIFICATION.md` (§§10–14), `MATHEMATICS.md` (result shapes). Eq. 1–2: success ⇒ result non-null and shape ∈ S_action; error ⇒ error non-null.

## 1. Repo root and Batch 1 spec (result shapes)

Define result-shape keys per action from ACTION_SPECIFICATION.md. No dependency on router code.

In [ ]:
from pathlib import Path
import json

repo_root = Path.cwd().resolve()
while not (repo_root / "scripts" / "ci" / "verify_caio_m365_contract.py").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

BATCH1_ACTIONS = ["list_groups", "get_group", "create_group", "list_group_members", "add_group_member"]
BATCH1_RESULT_SHAPES = {
    "list_groups": ["groups", "count"],
    "get_group": ["group"],
    "create_group": ["group_id", "display_name", "mail_nickname"],
    "list_group_members": ["members", "count"],
    "add_group_member": ["group_id", "member_id", "added"],
}
print("Batch 1 actions:", BATCH1_ACTIONS)
print("Result-shape keys:", BATCH1_RESULT_SHAPES)

## 2. Contract checks (Eq. 1 & 2)

Postcondition: ok ⇒ result non-null and shape in S_action; ¬ok ⇒ error non-null.

In [ ]:
def check_postcondition(res):
    """Eq. 1 & 2: ok⇒result non-null; ¬ok⇒error non-null."""
    ok = res.get("ok")
    if ok is True:
        if res.get("result") is None:
            return False, "ok=true but result is null"
        return True, ""
    if ok is False:
        if res.get("error") is None:
            return False, "ok=false but error is null"
        return True, ""
    return False, "missing or invalid 'ok' field"

def check_result_shape(action, result):
    """Result must have required keys for action."""
    if action not in BATCH1_RESULT_SHAPES:
        return True, ""
    for key in BATCH1_RESULT_SHAPES[action]:
        if key not in result:
            return False, f"missing key '{key}' in result for {action}"
    return True, ""

print("Contract check helpers defined.")

## 3. Batch 1 implementation (in-notebook, mock)

Implement the five actions in the notebook. Use mocks so the notebook runs without Graph credentials; same response shape as the future router.

In [ ]:
def execute_list_groups(params=None):
    params = params or {}
    return {"ok": True, "result": {"groups": [], "count": 0}}

def execute_get_group(params):
    gid = (params or {}).get("group_id") or (params or {}).get("id")
    if not gid:
        return {"ok": False, "error": "Missing group_id or id", "result": None}
    return {"ok": True, "result": {"group": {"id": gid, "displayName": "Test"}}}

def execute_create_group(params):
    p = params or {}
    if not p.get("display_name") or not p.get("mail_nickname"):
        return {"ok": False, "error": "Missing display_name or mail_nickname", "result": None}
    return {"ok": True, "result": {"group_id": "g-new", "display_name": p["display_name"], "mail_nickname": p["mail_nickname"]}}

def execute_list_group_members(params):
    gid = (params or {}).get("group_id") or (params or {}).get("id")
    if not gid:
        return {"ok": False, "error": "Missing group_id", "result": None}
    return {"ok": True, "result": {"members": [], "count": 0}}

def execute_add_group_member(params):
    p = params or {}
    if not p.get("group_id") or not p.get("member_id"):
        return {"ok": False, "error": "Missing group_id or member_id", "result": None}
    return {"ok": True, "result": {"group_id": p["group_id"], "member_id": p["member_id"], "added": True}}

BATCH1_EXECUTORS = {
    "list_groups": (execute_list_groups, None),
    "get_group": (execute_get_group, {"group_id": "g1"}),
    "create_group": (execute_create_group, {"display_name": "G1", "mail_nickname": "g1"}),
    "list_group_members": (execute_list_group_members, {"group_id": "g1"}),
    "add_group_member": (execute_add_group_member, {"group_id": "g1", "member_id": "u1"}),
}
print("Batch 1 executors (mock) defined.")

## 4. VERIFY: Run and assert Eq. 1 & 2 and result shape

For each Batch 1 action, run the executor and assert postcondition and result shape.

In [ ]:
details = {}
postcondition_pass = True
shape_pass = True

for action in BATCH1_ACTIONS:
    fn, args = BATCH1_EXECUTORS[action]
    res = fn(args) if args is not None else fn()
    pc_ok, pc_msg = check_postcondition(res)
    postcondition_pass = postcondition_pass and pc_ok
    shape_ok, shape_msg = True, ""
    if res.get(\"ok\") and res.get(\"result\"):
        shape_ok, shape_msg = check_result_shape(action, res["result"])
    shape_pass = shape_pass and shape_ok
    details[action] = {"postcondition": pc_ok, "postcondition_msg": pc_msg, "shape": shape_ok, "shape_msg": shape_msg}
    assert pc_ok, f"{action}: {pc_msg}"
    assert shape_ok, f"{action}: {shape_msg}"

print("VERIFY: All Batch 1 actions pass postcondition and result shape.")
print("Postcondition pass:", postcondition_pass)
print("Shape pass:", shape_pass)

## 5. Write verification artifact

Emit artifact for CI / traceability. Same pattern as `caio_m365_contract_verification.json`.

In [ ]:
artifact = {
    "batch": 1,
    "actions": BATCH1_ACTIONS,
    "postcondition_pass": postcondition_pass,
    "response_schema_pass": shape_pass,
    "details": details,
}
artifact_dir = repo_root / "configs" / "generated"
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_path = artifact_dir / "ma_batch1_verification.json"
with open(artifact_path, "w") as f:
    json.dump(artifact, f, indent=2)
print("Artifact written:", artifact_path)
assert postcondition_pass and shape_pass, "Batch 1 verification failed"
print("Batch 1 notebook verification PASSED.")